Run from the repository root or `notebooks/`. 

Reads existing shadow and population pickle files under `fpc_results/`.

Edit the settings cell to change input runs, seeds, calibration ratios, and output location. The final cell saves the figure when `SAVE_FIGURE` is true.


In [ ]:
from pathlib import Path
import pickle
import re
import numpy as np
import scipy.stats as st
from tueplots import bundles
import matplotlib.pyplot as plt
from cycler import cycler

rc = bundles.iclr2024(usetex=False)
# Match the source figure colors.
palette = [
    "#E69F00",  # Orange
    "#56B4E9",  # Sky blue
    "#000000",  # Black
    "#009E73",  # Bluish green
    "#F0E442",  # Yellow
    "#0072B2",  # Blue
    "#D55E00",  # Vermilion
    "#CC79A7",  # Purple

]
rc.update({
    "axes.prop_cycle": cycler(color=palette),
    "legend.frameon": False,
    "axes.grid": False,
})
from matplotlib.ticker import StrMethodFormatter


In [ ]:
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RESULTS_ROOT = ROOT / "fpc_results"
OUTPUT_PATH = ROOT / "plots" / "auc_survival_2ratio.pdf"
SAVE_FIGURE = True

SEEDS = [0, 42, 100]
auc_survival_runs = {"ADAM": "patch_camelyon", "TABPFN": "adult"}
PANEL_TITLES = ["(a) PatchCamelyon", "(b) Adult"]
SURVIVAL_MODEL = "TABPFN"
SURVIVAL_TITLE = r"(c) Adult ($N/N_+=0.5$)"
survival_ratio = 0.5
N = 1000
RATIOS = [i / 10 for i in range(1, 10)]
CALIBRATION_RATIOS = (0.1, 0.8)
N_PLUS_VALUES = [int(round(N / r)) for r in RATIOS]
M_SHADOW = 2048
M_POP = 2048
N_EVAL = 100

survival_styles = {
    "Population": ((0, (7, 2, 1.5, 2)), 1.5, 1.0),  # Dash–dot
    "Uncorrected": ((0, (3, 2)), 1.5, 1.0),          # Short dash
    "Simple FPC": ((0, (6, 3)), 2.0, 1.0),           # Long dash
    "Decomposed FPC": ("-", 2.2, 1.0),               # Solid
}

colors = {
    "Population": "#555555",
    "Uncorrected": "#D55E00",
    "Simple FPC": "#0072B2",
    "Decomposed FPC": "#CC79A7",
}


In [ ]:
class NumpyCompatUnpickler(pickle.Unpickler):
    def find_class(self, module, name):
        if module.startswith("numpy._core"):
            module = module.replace("numpy._core", "numpy.core", 1)
        return super().find_class(module, name)


def load_pickle(path):
    with open(path, "rb") as f:
        return NumpyCompatUnpickler(f).load()


def load_shadow_chunks(ratio, seed, results, dataset):
    """Reassemble all finite-frame model chunks in model-index order."""
    folder = (
        results / dataset / f"Seed={seed}" / f"N={N}" / f"ratio={ratio}"
    )
    files = sorted(
        # Exclude separate evaluation runs such as shadow_0_2048_eval_test.pkl.
        (p for p in folder.glob("shadow_*.pkl")
         if re.fullmatch(r"shadow_\d+_\d+\.pkl", p.name)),
        key=lambda p: int(re.search(r"shadow_(\d+)_", p.name).group(1)),
    )

    if not files:
        raise FileNotFoundError(f"No shadow chunks found in {folder}")

    stats = np.full((M_SHADOW, N_EVAL), np.nan, dtype=np.float64)
    membership = np.zeros((M_SHADOW, N_EVAL), dtype=bool)

    for path in files:
        d = load_pickle(path)
        start, stop = int(d["start_idx"]), int(d["stop_idx"])
        stats[start:stop] = d["target_stats"]
        membership[start:stop] = d["target_membership"]

    if np.isnan(stats).any():
        missing = np.where(np.isnan(stats).all(axis=1))[0]
        N_plus = int(round(N / ratio))
        raise ValueError(f"Missing shadow-model chunks for N_plus={N_plus}: {missing[:20]}")

    return stats, membership


def load_population_chunks(seed, results, dataset):
    """Reassemble the per-target population LOO matrix."""
    folder = results / dataset / f"Seed={seed}" / f"N={N}"
    files = list(folder.glob("population_targets_*_models_*.pkl"))

    if not files:
        raise FileNotFoundError(f"No population chunks found in {folder}")

    stats = np.full((N_EVAL, M_POP), np.nan, dtype=np.float64)
    membership = np.zeros((N_EVAL, M_POP), dtype=bool)

    for path in files:
        d = load_pickle(path)
        ts, te = int(d["target_start_idx"]), int(d["target_stop_idx"])
        ms, me = int(d["model_start_idx"]), int(d["model_stop_idx"])

        stats[ts:te, ms:me] = d["target_stats"]
        membership[ts:te, ms:me] = d["target_membership"]

    if np.isnan(stats).any():
        loc = np.argwhere(np.isnan(stats))
        raise ValueError(f"Population results are incomplete. First missing entries: {loc[:20]}")

    return stats, membership

def per_target_params_from_mixed(stats, membership):
    """stats/membership shape: models x targets."""
    n_targets = stats.shape[1]
    mu_in = np.empty(n_targets)
    mu_out = np.empty(n_targets)
    var_in = np.empty(n_targets)
    var_out = np.empty(n_targets)
    n_in = np.empty(n_targets, dtype=int)
    n_out = np.empty(n_targets, dtype=int)

    for j in range(n_targets):
        m = membership[:, j]
        s_in = stats[m, j]
        s_out = stats[~m, j]

        if len(s_in) < 2 or len(s_out) < 2:
            raise ValueError(f"Target {j} has too few IN/OUT observations.")

        mu_in[j] = s_in.mean()
        mu_out[j] = s_out.mean()
        var_in[j] = s_in.var(ddof=1)
        var_out[j] = s_out.var(ddof=1)
        n_in[j] = len(s_in)
        n_out[j] = len(s_out)

    return mu_in, mu_out, var_in, var_out, n_in, n_out


def per_target_params_population(stats, membership):
    """stats/membership shape: targets x models."""
    return per_target_params_from_mixed(stats.T, membership.T)

def lira_np_auc(mu_in, mu_out, var_in, var_out):
    mu_in = np.asarray(mu_in)
    mu_out = np.asarray(mu_out)
    var_in = np.asarray(var_in)
    var_out = np.asarray(var_out)

    delta = mu_in - mu_out
    S = var_in + var_out
    D = var_in - var_out

    auc = np.empty_like(delta, dtype=float)

    equal = np.abs(D) <= 1e-12 * S

    # Equal variances: LLR is linear.
    auc[equal] = st.norm.cdf(
        np.abs(delta[equal]) / np.sqrt(S[equal])
    )

    # Unequal variances: quadratic LLR.
    for i in np.where(~equal)[0]:
        z_u = -delta[i] / np.sqrt(S[i])
        z_v = -delta[i] * np.sqrt(S[i]) / abs(D[i])
        rho = abs(D[i]) / S[i]

        phi2 = st.multivariate_normal.cdf(
            [z_u, z_v],
            mean=[0.0, 0.0],
            cov=[[1.0, rho],
                 [rho, 1.0]],
        )

        auc[i] = (
            1.0
            - st.norm.cdf(z_u)
            - st.norm.cdf(z_v)
            + 2.0 * phi2
        )

    return auc

def fit_lambda(factors, variances):
    """Per-target decomposition using exactly two calibration ratios."""
    f_a, f_b = factors
    v_a, v_b = variances
    V1 = (v_a - v_b) / (f_a - f_b)
    V2 = (f_a * v_b - f_b * v_a) / (f_a - f_b)
    return np.clip(V2 / (V1 + V2), 0.0, 1.0)

def evaluate_gaussian_metrics(mu_in, mu_out, var_in, var_out):
    return {"AUC": {"LLR analytical": lira_np_auc(mu_in, mu_out, var_in, var_out)}}

def compute_seed_results(results, dataset, seeds=SEEDS):
    # Conditioned finite-population factors, shared across seeds.
    f_in_all = np.array([1.0 - (N - 1) / (Np - 1) for Np in N_PLUS_VALUES])
    f_out_all = np.array([1.0 - N / (Np - 1) for Np in N_PLUS_VALUES])
    calibration_indices = [RATIOS.index(r) for r in CALIBRATION_RATIOS]
    calibration_n_plus = [N_PLUS_VALUES[i] for i in calibration_indices]
    results_by_seed = {}
    lambda_by_seed = {}
    for seed in seeds:
        pop_stats, pop_membership = load_population_chunks(seed, results, dataset)
        mu_in_pop, mu_out_pop, var_in_pop, var_out_pop, _, _ = (
            per_target_params_population(pop_stats, pop_membership)
        )
        population = evaluate_gaussian_metrics(mu_in_pop, mu_out_pop, var_in_pop, var_out_pop)
        finite_frame = {}
        fpc = {}
        params_by_ratio = {}
        for ratio, N_plus in zip(RATIOS, N_PLUS_VALUES):
            stats, membership = load_shadow_chunks(ratio, seed, results, dataset)
            mu_in, mu_out, var_in, var_out, _, _ = per_target_params_from_mixed(stats, membership)
            params_by_ratio[N_plus] = (mu_in, mu_out, var_in, var_out)
            finite_frame[N_plus] = evaluate_gaussian_metrics(mu_in, mu_out, var_in, var_out)
            f_in = 1.0 - (N - 1) / (N_plus - 1)
            f_out = 1.0 - N / (N_plus - 1)
            fpc[N_plus] = evaluate_gaussian_metrics(mu_in, mu_out, var_in / f_in, var_out / f_out)
        # Solve IN and OUT components from the two calibration ratios within this seed.
        # Population estimates are not used in either fit.
        lambda_in = fit_lambda(
            f_in_all[calibration_indices],
            np.stack([params_by_ratio[Np][2] for Np in calibration_n_plus]),
        )
        lambda_out = fit_lambda(
            f_out_all[calibration_indices],
            np.stack([params_by_ratio[Np][3] for Np in calibration_n_plus]),
        )
        lambda_by_seed[seed] = {"IN": lambda_in, "OUT": lambda_out}
        lambda_fpc = {}
        for N_plus, f_in, f_out in zip(N_PLUS_VALUES, f_in_all, f_out_all):
            mu_in, mu_out, var_in, var_out = params_by_ratio[N_plus]
            var_in_corrected = var_in / (f_in * (1.0 - lambda_in) + lambda_in)
            var_out_corrected = var_out / (f_out * (1.0 - lambda_out) + lambda_out)
            lambda_fpc[N_plus] = evaluate_gaussian_metrics(
                mu_in, mu_out, var_in_corrected, var_out_corrected,
            )
        results_by_seed[seed] = {
            "Population": population,
            "Uncorrected": finite_frame,
            "Simple FPC": fpc,
            "Decomposed FPC": lambda_fpc,
        }
    
    return results_by_seed, lambda_by_seed

In [ ]:
auc_survival_results = {
    model: compute_seed_results(RESULTS_ROOT / model, dataset, seeds=SEEDS)[0]
    for model, dataset in auc_survival_runs.items()
}
survival_N_plus = int(round(N / survival_ratio))


In [ ]:
with plt.rc_context(rc):
    fig, axes = plt.subplots(1, 3, figsize=(rc["figure.figsize"][0], 2.65),
                             layout="constrained")
    axes[1].sharey(axes[0])
    for ax, model, title in zip(
        axes[:2], list(auc_survival_runs), PANEL_TITLES,
    ):
        by_seed = auc_survival_results[model]
        for label, color in colors.items():
            seed_means = np.array([
                [(by_seed[seed][label] if label == "Population"
                  else by_seed[seed][label][Np])["AUC"]["LLR analytical"].mean()
                 for Np in N_PLUS_VALUES]
                for seed in SEEDS
            ])
            linestyle, linewidth, alpha = survival_styles[label]
            ax.fill_between(RATIOS, seed_means.min(axis=0), seed_means.max(axis=0),
                            color=color, alpha=0.12)
            ax.plot(RATIOS, np.median(seed_means, axis=0), color=color,
                    linestyle=linestyle, lw=1., alpha=alpha, label=label)
        ax.set_title(title)
        ax.set_xlabel(r"$N/N_+$")
        ax.set_xticks(RATIOS[::2])
        ax.set_yscale("linear")
        ax.yaxis.set_major_formatter(StrMethodFormatter("{x:.2f}"))
        ax.set_box_aspect(1)
    axes[0].set_ylabel("Mean AUC")
    axes[1].tick_params(labelleft=False)

    ax = axes[2]
    by_seed = auc_survival_results[SURVIVAL_MODEL]
    for label, color in colors.items():
        seed_values = [
            (by_seed[seed][label] if label == "Population"
             else by_seed[seed][label][survival_N_plus])["AUC"]["LLR analytical"]
            for seed in SEEDS
        ]
        pooled_auc = np.sort(np.concatenate(seed_values))
        thresholds = np.unique(np.concatenate(([0.5, 1.0], pooled_auc)))
        pooled_survival = (
            len(pooled_auc) - np.searchsorted(pooled_auc, thresholds, side="left")
        ) / len(pooled_auc)
        linestyle, linewidth, alpha = survival_styles[label]
        ax.step(thresholds, pooled_survival, where="pre", color=color,
                linestyle=linestyle, lw=1.0, alpha=alpha, label=label)
    ax.set_title(SURVIVAL_TITLE)
    ax.set_xlabel("AUC threshold")
    ax.set_ylabel(r"$1-$ CDF")
    ax.set_yscale("log")
    ax.set_xlim(0.5, 1.0)
    # Match the source figure: display survival probabilities of at least 1%.
    ax.set_ylim(1e-2, 1.0)
    ax.set_box_aspect(1)
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="outside lower center", ncol=4, frameon=False)
    if SAVE_FIGURE:
        fig.savefig(OUTPUT_PATH, bbox_inches="tight")
    plt.show()
